# Risk Agent

Runs the full pipeline: **Goal -> Plan -> Retrieve -> Analyze -> Decide -> Recommend -> Validate**.

In [1]:
import sys
from pathlib import Path

import faiss
import pandas as pd
from sentence_transformers import SentenceTransformer

def _locate_and_import_agent_core():
    here = Path.cwd()
    candidates = [here, here / "agents", here.parent / "agents", here.parent]
    for c in candidates:
        if (c / "agent_core.py").exists():
            sys.path.insert(0, str(c))
            import agent_core as agent_core_module
            return agent_core_module
    raise FileNotFoundError(
        "Could not locate agent_core.py. Looked in: "
        + ", ".join(str(c) for c in candidates)
        + f". Current working directory: {here}"
    )


core = _locate_and_import_agent_core()

DATA_DIR = core.find_data_dir()
CHUNKS_PATH   = DATA_DIR / "chunked_data.json"
INDEX_PATH    = DATA_DIR / "sap_intelligence.index"
OUT_PATH      = DATA_DIR / "risks.json"
PIPELINE_PATH = DATA_DIR / "risks_pipeline.json"

EMBED_MODEL = "BAAI/bge-small-en-v1.5"

GOAL = (
    "Identify 3 to 5 concrete current business risks for SAP - "
    "competitive threats, regulatory changes, negative sentiment, or "
    "supply chain issues - using the indexed SAP news corpus as evidence."
)

print("data dir:", DATA_DIR)


data dir: /home/jovyan/vault/Testing/Testing/notebook/data


In [2]:
chunks_df = pd.read_json(CHUNKS_PATH)
index = faiss.read_index(str(INDEX_PATH))
embed_model = SentenceTransformer(EMBED_MODEL)

print("chunks loaded:", len(chunks_df))
print("vectors in index:", index.ntotal)

core.warmup()


chunks loaded: 1400
vectors in index: 1400
warming up model...


warmup done in 0.5 sec


In [3]:
def retrieve_news(query, k=6):
    q_vec = embed_model.encode([query], normalize_embeddings=True).astype("float32")
    _, idx = index.search(q_vec, k)
    return chunks_df.iloc[idx[0]]["text"].tolist()


def read_previous_findings(_args=None):
    previous = core.load_previous_output(OUT_PATH)
    if not previous:
        return "No previous findings on file - this looks like the first run."
    return previous


RETRIEVAL_TOOL_HANDLERS = {
    "retrieve_news": lambda args: retrieve_news(args.get("query", ""), int(args.get("k", 6) or 6)),
    "read_previous_findings": lambda args: read_previous_findings(args),
}

RETRIEVAL_TOOL_SCHEMAS = [
    {
        "type": "function",
        "function": {
            "name": "retrieve_news",
            "description": (
                "Semantic search over the indexed SAP news corpus. Call this "
                "with your own queries to investigate different angles - "
                "competitive threats, regulatory issues, negative sentiment, "
                "supply chain, etc."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "Natural language search query"},
                    "k": {"type": "integer", "description": "Number of chunks to retrieve (default 6)"},
                },
                "required": ["query"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "read_previous_findings",
            "description": (
                "Read the risks you identified in the previous run, if any, "
                "so you can check what is still supported by current evidence."
            ),
            "parameters": {"type": "object", "properties": {}},
        },
    },
]


In [4]:
DECIDE_TOOL_SCHEMA = {
    "type": "function",
    "function": {
        "name": "finalize_risks",
        "description": "Call this exactly once to submit your final list of business risks.",
        "parameters": {
            "type": "object",
            "properties": {
                "risks": {
                    "type": "array",
                    "description": "3 to 5 concrete business risks",
                    "items": {
                        "type": "object",
                        "properties": {
                            "title": {"type": "string"},
                            "risk_category": {"type": "string", "description": "Competitive, Regulatory, Sentiment, or Supply Chain"},
                            "severity_level": {"type": "string", "description": "High, Medium, or Low"},
                            "evidence": {"type": "string"},
                            "confidence_score": {"type": "integer", "description": "0 to 100"},
                        },
                        "required": ["title", "risk_category", "severity_level", "evidence", "confidence_score"],
                    },
                }
            },
            "required": ["risks"],
        },
    },
}

VALIDATE_TOOL_SCHEMA = {
    "type": "function",
    "function": {
        "name": "submit_validated_risks",
        "description": "Call this exactly once to submit your evidence-checked, final list of risks.",
        "parameters": {
            "type": "object",
            "properties": {
                "validated_risks": {
                    "type": "array",
                    "items": {
                        "type": "object",
                        "properties": {
                            "title": {"type": "string"},
                            "risk_category": {"type": "string"},
                            "severity_level": {"type": "string"},
                            "evidence": {"type": "string"},
                            "confidence_score": {"type": "integer"},
                            "validation_status": {"type": "string", "description": "Supported, Revised, or Unsupported"},
                            "validation_notes": {"type": "string"},
                        },
                        "required": ["title", "risk_category", "severity_level", "evidence", "confidence_score", "validation_status", "validation_notes"],
                    },
                }
            },
            "required": ["validated_risks"],
        },
    },
}


In [5]:
validated_risks, pipeline = core.run_full_pipeline(
    goal_description=GOAL,
    retrieval_tool_schemas=RETRIEVAL_TOOL_SCHEMAS,
    retrieval_tool_handlers=RETRIEVAL_TOOL_HANDLERS,
    analyze_tool_schema=core.make_analyze_tool_schema("risks"),
    decide_tool_schema=DECIDE_TOOL_SCHEMA,
    decide_tool_name="finalize_risks",
    decide_items_key="risks",
    validate_tool_schema=VALIDATE_TOOL_SCHEMA,
    validate_tool_name="submit_validated_risks",
    validate_items_key="validated_risks",
)

print()
print("=" * 70)
print(len(validated_risks), "validated risks:")
for r in validated_risks:
    print("-", r.get("title"), "|", r.get("risk_category"), "|", r.get("validation_status"))



STAGE 1: PLAN


PLAN: {
  "goal": "Identify current business risks facing SAP based on news articles.",
  "planned_steps": [
    "Search the indexed SAP news corpus for competitive threats and regulatory changes",
    "Analyze news articles for negative sentiment towards SAP",
    "Identify potential supply chain issues affecting SAP through news reports"
  ]
}

STAGE 2: RETRIEVE


  [retrieve step 1] called `retrieve_news` with {"k": "6", "query": "SAP competitive threats and regulatory changes"}


  [retrieve step 2] agent signalled done retrieving

STAGE 3: ANALYZE


ANALYSIS: {
  "candidate_items": "[\"Regulatory Changes Impacting SAP\", \"Supply Chain Disruptions Affecting SAP\", \"Negative Sentiment towards SAP\"]",
  "key_observations": [
    "There are news articles discussing regulatory changes and their impact on SAP.",
    "European leaders view AI infrastructure as a foundation of economic security, enabling compliance with stringent regulations.",
    "Businesses are turning to data federation and cloud-based solutions to extend visibility and instill resilience across their supply chains."
  ]
}

STAGE 4: DECIDE + RECOMMEND


DRAFT DECISION: {
  "risks": "[{\"title\": \"Regulatory Changes Impacting SAP\", \"risk_category\": \"Regulatory\", \"severity_level\": \"High\", \"evidence\": \"European leaders view AI infrastructure as a foundation of economic security, enabling compliance with stringent regulations.\", \"confidence_score\": 90}, {\"title\": \"Supply Chain Disruptions Affecting SAP\", \"risk_category\": \"Supply Chain\", \"severity_level\": \"Medium\", \"evidence\": \"Businesses are turning to data federation and cloud-based solutions to extend visibility and instill resilience across their supply chains.\", \"confidence_score\": 85}, {\"title\": \"Negative Sentiment towards SAP\", \"risk_category\": \"Sentiment\", \"severity_level\": \"Low\", \"evidence\": \"There are news articles discussing regulatory changes and their impact on SAP.\", \"confidence_score\": 70}]"
}

STAGE 5: VALIDATE


VALIDATED: {
  "validated_risks": [
    {
      "confidence_score": 90,
      "evidence": "European leaders view AI infrastructure as a foundation of economic security, enabling compliance with stringent regulations.",
      "risk_category": "Regulatory",
      "severity_level": "High",
      "title": "Regulatory Changes Impacting SAP",
      "validation_notes": "",
      "validation_status": "Supported"
    },
    {
      "confidence_score": 85,
      "evidence": "Businesses are turning to data federation and cloud-based solutions to extend visibility and instill resilience across their supply chains.",
      "risk_category": "Supply Chain",
      "severity_level": "Medium",
      "title": "Supply Chain Disruptions Affecting SAP",
      "validation_notes": "",
      "validation_status": "Supported"
    },
    {
      "confidence_score": 70,
      "evidence": "There are news articles discussing regulatory changes and their impact on SAP.",
      "risk_category": "Sentiment",
      "sev

In [6]:
core.save_json(OUT_PATH, validated_risks)
core.save_json(PIPELINE_PATH, pipeline)


saved to /home/jovyan/vault/Testing/Testing/notebook/data/risks.json
saved to /home/jovyan/vault/Testing/Testing/notebook/data/risks_pipeline.json
